! pip install langchain_groq

In [46]:
import os 
from dotenv import load_dotenv
load_dotenv()

gpk =os.getenv("GPK")
#print(gpk)

In [47]:
from langchain_groq import ChatGroq

model= ChatGroq(model ="gemma2-9b-it" , groq_api_key = gpk)

In [ ]:
from langchain_core.messages import HumanMessage
op= model.invoke([HumanMessage(content = "Hi , My name is nithin and working as Data analyst")])
op

In [4]:
from langchain_core.messages import HumanMessage, AIMessage

response = model.invoke(
    [
        HumanMessage(content="Hi, My name is Nithin and working as Data analyst"),
        AIMessage(content="""Hello Nithin!\n\nIt's great to meet you. That's awesome that you're working as a Data Analyst.
        \n\nIs there anything I can help you with today?"""),
        HumanMessage(content="What is my name?")
    ]
)
#response
print(response.content)

Your name is Nithin. 😊

I remember that from our introduction!  Is there anything else I can help you with?



In [5]:
#pip install langchain_community

In [6]:
from langchain_community.chat_message_histories import ChatMessageHistory 
from langchain_core.chat_history import BaseChatMessageHistory 
from langchain_core.runnables.history import RunnableWithMessageHistory

store ={}

def get_ses(session_id:str)->BaseChatMessageHistory:
    if session_id not in store:
        store[session_id] = ChatMessageHistory()
    return store[session_id]

with_message_history = RunnableWithMessageHistory(model , get_ses)

In [7]:
config = {"configurable":{"session_id":"chat1"}}

In [8]:
resp = with_message_history.invoke(
        [ HumanMessage(content="Hi, My name is Nithin and working as Data analyst") ],
    config = config )

print(resp.content)

Hi Nithin, it's nice to meet you!  

It's great that you're a data analyst. That's a really interesting and important field. 

What kind of data do you typically work with? What are some of the projects you've enjoyed working on?



In [9]:
resp = with_message_history.invoke(
        [ HumanMessage(content="What is my name ") ],
    config = config )

print(resp.content)

Your name is Nithin.  

I remembered from our earlier conversation.  😊  How can I help you today?



In [10]:
config = {"configurable":{"session_id":"chat2"}}

In [11]:
resp = with_message_history.invoke(
        [ HumanMessage(content="What is my name ") ],
    config = config )

print(resp.content)

As a large language model, I have no memory of past conversations and do not know your name. If you'd like to tell me your name, I'd be happy to remember it for our current interaction.



# Prompt Templates

In [12]:
from langchain_core.prompts import ChatPromptTemplate , MessagesPlaceholder

prompt =ChatPromptTemplate.from_messages(    
    [ ( "system" , "Your are a helpfull assistant. Answer all the questions" ),
      MessagesPlaceholder(variable_name="messages")
       ] )

chain = prompt | model

In [13]:
op = chain.invoke({"messages": [HumanMessage(content="What is my name ") ]})
op.content

"As I have no memory of past conversations and I don't have access to personal information about you, I don't know your name.  \n\nIf you'd like to tell me your name, I'd be happy to use it! 😊\n\n"

In [14]:
with_message = RunnableWithMessageHistory(chain  , get_ses)

In [15]:
config = {"configurable":{"session_id":"chat3"}}

In [16]:
respose =with_message.invoke(
     [HumanMessage(content="Hi, My name is Nithin and working as Data analyst") ],
    config =config)

print(respose.content)
    

Hi Nithin, it's nice to meet you!

I'm glad you're here.  As a helpful assistant, I'm ready to answer your questions and assist you in any way I can.  

What can I help you with today?  

Since you're a data analyst, perhaps you have some data-related questions or tasks I can help with?  I can help with things like:

* **Understanding data:** I can help you interpret data, identify patterns, and draw insights.
* **Data cleaning and preparation:** I can help you clean and prepare your data for analysis.
* **Data visualization:** I can help you create visualizations to communicate your findings.
* **Code generation:** I can generate code in various programming languages to help you with your data analysis tasks.


Let me know how I can be of service! 




In [17]:
from langchain_core.prompts import ChatPromptTemplate , MessagesPlaceholder

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant. Always reply in the {language} language."),
    MessagesPlaceholder(variable_name="messages")
])

chain = prompt | model

op = chain.invoke({
    "messages": [HumanMessage(content= "Hi, My name is Nithin and working as Data analyst" )],
    "language": "Kannada"
})

print( op.content)

ನೀನು ನಿಧಿನ್ ಎಂದು ಹೇಳಿದ್ದೀಯಾ? ಡೇಟಾ ಅನಾಲಿಸ್ಟ್‌ ಆಗಿ ಕೆಲಸ ಮಾಡುತ್ತಿದ್ದೀಯಾ? 




In [18]:
with_message_history =RunnableWithMessageHistory(
    chain ,
    get_ses , 
    input_messages_key="messages" )

config = {"configurable":{"session_id":"chat4"}}

op = with_message_history.invoke({
    "messages": [HumanMessage(content= "Hi, My name is Nithin and working as Data analyst" )],
    "language": "english" } ,  config = config)
print(op.content)

op = with_message_history.invoke({
    "messages": [HumanMessage(content= "What is my name" )],
    "language": "english" } ,  config = config)
print(op.content)


op = with_message_history.invoke({
    "messages": [HumanMessage(content= "What is name date of birth" )],
    "language": "english" } ,  config = config )
print(op.content)

op = with_message_history.invoke({
    "messages": [HumanMessage(content= "Hi, My name is Raj" )],
    "language": "english" } ,  config = config)
print(op.content)

op = with_message_history.invoke({
    "messages": [HumanMessage(content= "What is my name" )],
    "language": "english" } ,  config = config)
print(op.content)


Hello Nithin, it's nice to meet you!  

It's great that you're working as a data analyst. That's a fascinating field. 

Is there anything I can help you with today related to your work? 


Your name is Nithin.  😊 

As an AI, I don't have access to personal information like names and dates of birth unless you choose to share it with me. 

Is there anything else I can help you with?

Hi Raj, it's nice to meet you!  

What can I do for you today? 😊 

Your name is Raj.  

How can I help you further?  😊   



# Managing the Chat Conversation History Using Langchain

In [19]:
#!pip install transformers

In [34]:
from langchain_core.messages import trim_messages , SystemMessage 

trim =trim_messages(
    max_tokens = 50 ,
    strategy ="last" ,
    token_counter = model , 
    include_system = True , 
    allow_partial = False , 
    start_on ="human"
)

messages =[
    SystemMessage(content="you're a good assissnt "),
    HumanMessage(content="hi!, I am nithin") ,
    AIMessage(content ="Hi"),
    HumanMessage(content="2+4") ,
    AIMessage(content ="6"),
    HumanMessage(content="What is today day") ,
    AIMessage(content ="Thurday"),
    HumanMessage(content="What is today date") ,
    AIMessage(content ="23-04-2023"),
    HumanMessage(content="Any girl name") ,
    AIMessage(content ="Daridhra"),
    HumanMessage(content="best day ?") ,
    AIMessage(content ="No"),
    HumanMessage(content="Why") ,
    AIMessage(content ="IDK"),
    HumanMessage(content="Why why ") ,
    AIMessage(content ="go and talk to her"),
    HumanMessage(content=" with whom ") ,
    AIMessage(content ="Chikkamma"),
    HumanMessage(content="What she is not looking good") ,
    AIMessage(content ="she more preaty"),
]
trim.invoke(messages)

[SystemMessage(content="you're a good assissnt ", additional_kwargs={}, response_metadata={}),
 HumanMessage(content='Why why ', additional_kwargs={}, response_metadata={}),
 AIMessage(content='go and talk to her', additional_kwargs={}, response_metadata={}),
 HumanMessage(content=' with whom ', additional_kwargs={}, response_metadata={}),
 AIMessage(content='Chikkamma', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='What she is not looking good', additional_kwargs={}, response_metadata={}),
 AIMessage(content='she more preaty', additional_kwargs={}, response_metadata={})]

In [41]:
from operator import itemgetter
from langchain_core.runnables import RunnablePassthrough


trim =trim_messages(
    max_tokens = 100 ,
    strategy ="last" ,
    token_counter = model , 
    include_system = True , 
    allow_partial = False , 
    start_on ="human"
)

chain = (
    RunnablePassthrough.assign(messages=itemgetter("messages")|trim )
    | prompt
    | model )

response =  chain.invoke(
    {
        "messages":messages+[HumanMessage(content="What is my name")] ,
        "language":"English"
    }
)
print(response.content)
        

As an AI, I don't have access to personal information about you, including your name.  

If you'd like to tell me your name, I'd be happy to know! 😊  




In [44]:
# Let wrap 

with_message_history =RunnableWithMessageHistory(
    chain ,
    get_ses , 
    input_messages_key="messages" )

config = {"configurable":{"session_id":"chat5"}}

op = with_message_history.invoke({
    "messages": [HumanMessage(content= "What is my name" )],
    "language": "english" } ,  config = config)
print(op.content)


As an AI, I have no memory of past conversations and do not know your name.  

If you'd like to tell me your name, I'd be happy to use it! 😊  


